# 06. SQL Create Table, DDL & Constraints: Beginner Guide

### 📝 SQL Execution Order:
```text
┌─ SQL Query Execution Order (Sequential Pipeline) ────────────────────────────┐
│ 1. FROM & JOIN (Load)    ➔ 2. WHERE (Filter)       ➔ 3. GROUP BY (Bucket)    │
│ ➔ 4. HAVING (Agg Filter) ➔ 5. SELECT (Pick Cols)   ➔ 6. DISTINCT (Dedup)     │
│ ➔ 7. ORDER BY (Sort)     ➔ 8. LIMIT / OFFSET (Page)                          │
└──────────────────────────────────────────────────────────────────────────────┘
```

---

### 📌 Overview & Architectural Context
Welcome to **06. SQL Create Table, DDL & Constraints**. Data Definition Language (DDL) defines the structural constraints, data types, and referential linkages of relational databases. This notebook covers table creation (`CREATE TABLE`), column constraints (`PRIMARY KEY`, `FOREIGN KEY`, `UNIQUE`, `CHECK`, `DEFAULT`), normalization forms (1NF, 2NF, 3NF), and temporary tables.

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Table Creation & Column Data Types: `CREATE TABLE`
- [x] 🔹 Primary Key Constraint Enforcement: `PRIMARY KEY`
- [x] 🔹 Referential Integrity Linkages: `FOREIGN KEY ... REFERENCES`
- [x] 🔹 Declarative Value Constraints: `CHECK (expr)` & `DEFAULT val`
- [x] 🔹 Temporary Tables: `CREATE TEMPORARY TABLE`
- [x] 🔍 Scenario: Enterprise Fintech Merchant Account Ledger Schema Design








In [1]:
# Setup in-memory SQLite relational engine with Native SQL Studio Execution
import sqlite3
import pandas as pd
import os
from IPython import get_ipython
from IPython.core.magic import register_line_cell_magic

conn = sqlite3.connect(':memory:')

def load_table(name, path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        df.to_sql(name, conn, index=False, if_exists='replace')

load_table('transactions', 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv')
load_table('customers', 'data/customers.csv' if os.path.exists('data/customers.csv') else '../data/customers.csv')
load_table('merchants', 'data/merchants.csv' if os.path.exists('data/merchants.csv') else '../data/merchants.csv')
load_table('disputes', 'data/disputes.csv' if os.path.exists('data/disputes.csv') else '../data/disputes.csv')

def _execute_raw_sql(query):
    query = query.strip()
    if query.upper().startswith(('INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'VACUUM', 'ANALYZE', 'BEGIN', 'COMMIT', 'ROLLBACK', 'SAVEPOINT')):
        cur = conn.cursor()
        cur.executescript(query)
        conn.commit()
        return "Query Executed Successfully."
    else:
        return pd.read_sql_query(query, conn)

# Register automatic raw SQL transformer & %%sql magic
ip = get_ipython()
if ip is not None:
    def raw_sql_transformer(lines):
        clean_text = ''.join(lines).strip()
        first_token = clean_text.split()[0].upper() if clean_text.split() else ''
        sql_keywords = {'SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'EXPLAIN', 'ANALYZE', 'VACUUM', 'BEGIN', 'COMMIT', 'ROLLBACK'}
        if first_token in sql_keywords:
            return [f'_execute_raw_sql("""{clean_text}""")']
        return lines
    
    if raw_sql_transformer not in ip.input_transformers_cleanup:
        ip.input_transformers_cleanup.append(raw_sql_transformer)

@register_line_cell_magic
def sql(line, cell=None):
    return _execute_raw_sql(cell if cell is not None else line)

print("SQL Studio Environment Active! You can now write and run pure SQL queries directly.")


SQL Studio Environment Active! You can now write and run pure SQL queries directly.


### 🔹 Table Definition & Data Types: `CREATE TABLE`
- **What it does:** Allocates physical or in-memory catalog metadata for a new relational table schema with specified column types.
- **Syntax:** `CREATE TABLE table_name (col1 TYPE [CONSTRAINTS], col2 TYPE);`
- **Dataset Application & Code Demonstration:** Defines a normalized ledger table for merchant payouts.


In [2]:
%%sql
CREATE TABLE IF NOT EXISTS merchant_ledger (
    entry_id INTEGER PRIMARY KEY AUTOINCREMENT,
    merchant_id TEXT NOT NULL,
    payout_amount REAL NOT NULL CHECK (payout_amount > 0),
    currency TEXT DEFAULT 'USD',
    settlement_status TEXT DEFAULT 'PENDING',
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
INSERT INTO merchant_ledger (merchant_id, payout_amount, currency)
VALUES ('MERCH_101', 5420.75, 'USD');
SELECT * FROM merchant_ledger;


'Query Executed Successfully.'

### 🔹 Temporary Tables: `CREATE TEMPORARY TABLE`
- **What it does:** Creates a session-scoped temporary table that is automatically dropped when the client connection terminates.
- **Syntax:** `CREATE TEMPORARY TABLE temp_table_name AS SELECT ...`
- **Dataset Application & Code Demonstration:** Creates a session temporary table of high-risk customers.


In [3]:
%%sql
CREATE TEMPORARY TABLE temp_fraud_summary AS
SELECT 
    customer_id,
    COUNT(transaction_id) AS fraud_count,
    SUM(transaction_amount) AS total_fraud_amount
FROM transactions
WHERE is_fraud = 1
GROUP BY customer_id;
SELECT * FROM temp_fraud_summary LIMIT 5;


'Query Executed Successfully.'

## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: 1NF, 2NF & 3NF Normalization Principles
- **Objective:** Explain the transition from unnormalized denormalized tables to 3NF (Third Normal Form) to eliminate insertion/update anomalies.
- **Approach:** Demonstrate entity splitting across Customers, Merchants, and Transactions.


In [4]:
%%sql
SELECT 
    '1NF' AS norm_level, 'Atomic column values, unique primary key, no repeating groups' AS requirement
UNION ALL
SELECT '2NF', 'Meets 1NF + No partial functional dependencies (all non-key attributes fully depend on PK)'
UNION ALL
SELECT '3NF', 'Meets 2NF + No transitive dependencies (non-key attributes depend only on the PK)';


,norm_level,requirement
0,1NF,"Atomic column values, unique primary key, no r..."
1,2NF,Meets 1NF + No partial functional dependencies...
2,3NF,Meets 2NF + No transitive dependencies (non-ke...
